In [ ]:
import json
import os
import shutil
import random

# 1. Paths Configuration
seaclear_dir = r"D:\01_University_&_Study\Samsung Innovation Campus - Immersive with MiSK Skills\CapstoneProject\MarineGuard_Datasets\SeaClear\Seaclear Marine Debris Dataset"
json_path = os.path.join(seaclear_dir, "dataset.json")
output_dir = r"D:\01_University_&_Study\Samsung Innovation Campus - Immersive with MiSK Skills\CapstoneProject\MarineGuard_Datasets\MarineGuard_YOLO_Ready"

for split in ['train', 'val']:
    os.makedirs(os.path.join(output_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels', split), exist_ok=True)

# 2. Index Images by Site/Camera Group (Data Leakage Prevention)
image_info_map = {}
for root, dirs, files in os.walk(seaclear_dir):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(root, file)
            group_id = os.path.relpath(root, seaclear_dir)
            image_info_map[file] = {'path': full_path, 'group': group_id}

# 3. Class Mapping & Hard-Negatives
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

target_mapping = {'net_plastic': 0, 'rope_fiber': 1, 'rope_plastic': 1}
hard_negative_names = {'plant', 'branch_wood', 'cable_metal', 'rov_cable'}

cat_id_to_yolo = {}
hard_neg_cat_ids = set()

for cat in data['categories']:
    if cat['name'] in target_mapping:
        cat_id_to_yolo[cat['id']] = target_mapping[cat['name']]
    elif cat['name'] in hard_negative_names:
        hard_neg_cat_ids.add(cat['id'])

# 4. COCO to YOLO Normalization
def convert_coco_to_yolo(bbox, img_w, img_h):
    x_min, y_min, w, h = bbox
    x_center = (x_min + (w / 2.0)) / img_w
    y_center = (y_min + (h / 2.0)) / img_h
    norm_w = w / img_w
    norm_h = h / img_h
    
    return (
        max(0.0, min(1.0, x_center)),
        max(0.0, min(1.0, y_center)),
        max(0.0, min(1.0, norm_w)),
        max(0.0, min(1.0, norm_h))
    )

# 5. Aggregate Positive & Hard-Negative Samples
img_annotations = {}
hard_negative_images = set()

for ann in data['annotations']:
    img_id = ann['image_id']
    cat_id = ann['category_id']
    
    if cat_id in cat_id_to_yolo:
        img_annotations.setdefault(img_id, []).append((cat_id_to_yolo[cat_id], ann['bbox']))
    elif cat_id in hard_neg_cat_ids:
        hard_negative_images.add(img_id)

pure_hard_negatives = list(hard_negative_images - set(img_annotations.keys()))
random.seed(42)
selected_hard_negs = random.sample(pure_hard_negatives, min(300, len(pure_hard_negatives)))
all_selected_img_ids = set(img_annotations.keys()).union(set(selected_hard_negs))

# 6. Site-Aware Split (80% Train / 20% Val)
all_groups = list(set(info['group'] for info in image_info_map.values()))
random.shuffle(all_groups)

split_idx = int(len(all_groups) * 0.8)
train_groups = set(all_groups[:split_idx])

images_meta = {img['id']: img for img in data['images']}
stats = {'train_pos': 0, 'train_neg': 0, 'val_pos': 0, 'val_neg': 0}

for img_id in all_selected_img_ids:
    meta = images_meta[img_id]
    file_name = os.path.basename(meta['file_name'])
    info = image_info_map.get(file_name)
    if not info:
        continue
        
    split = 'train' if info['group'] in train_groups else 'val'
    dest_img_path = os.path.join(output_dir, 'images', split, file_name)
    shutil.copy2(info['path'], dest_img_path)
    
    txt_name = os.path.splitext(file_name)[0] + '.txt'
    dest_txt_path = os.path.join(output_dir, 'labels', split, txt_name)
    
    anns = img_annotations.get(img_id, [])
    with open(dest_txt_path, 'w', encoding='utf-8') as f:
        for yolo_cls, bbox in anns:
            xc, yc, nw, nh = convert_coco_to_yolo(bbox, meta['width'], meta['height'])
            f.write(f"{yolo_cls} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}\n")
            
    if len(anns) > 0:
        stats[f'{split}_pos'] += 1
    else:
        stats[f'{split}_neg'] += 1

print("SeaClear dataset processed successfully:")
print(f"- Train: {stats['train_pos']} positive + {stats['train_neg']} hard-negative images")
print(f"- Val:   {stats['val_pos']} positive + {stats['val_neg']} hard-negative images")
print(f"Output directory: {output_dir}")

In [ ]:
import json
import os
import shutil

# 1. Paths Configuration
trashcan_base = r"D:\01_University_&_Study\Samsung Innovation Campus - Immersive with MiSK Skills\CapstoneProject\MarineGuard_Datasets\TrashCan\dataset\dataset\instance_version"
output_dir = r"D:\01_University_&_Study\Samsung Innovation Campus - Immersive with MiSK Skills\CapstoneProject\MarineGuard_Datasets\MarineGuard_YOLO_Ready"

os.makedirs(os.path.join(output_dir, 'images', 'test'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'labels', 'test'), exist_ok=True)

# 2. Class Standardization (0 = Net, 1 = Rope)
trashcan_mapping = {'trash_net': 0, 'trash_rope': 1}
json_files = [
    (os.path.join(trashcan_base, 'instances_train_trashcan.json'), os.path.join(trashcan_base, 'train')),
    (os.path.join(trashcan_base, 'instances_val_trashcan.json'), os.path.join(trashcan_base, 'val'))
]

test_images_count = 0
test_boxes_count = 0

# 3. Extract & Convert Annotations to YOLO Format
for json_file, img_folder in json_files:
    with open(json_file, 'r', encoding='utf-8') as f:
        t_data = json.load(f)
        
    cat_id_to_yolo = {
        cat['id']: trashcan_mapping[cat['name']]
        for cat in t_data['categories']
        if cat['name'] in trashcan_mapping
    }
    
    img_anns = {}
    for ann in t_data['annotations']:
        if ann['category_id'] in cat_id_to_yolo:
            img_anns.setdefault(ann['image_id'], []).append((cat_id_to_yolo[ann['category_id']], ann['bbox']))
            
    images_meta = {img['id']: img for img in t_data['images']}
    
    for img_id, anns in img_anns.items():
        meta = images_meta[img_id]
        file_name = meta['file_name']
        src_img_path = os.path.join(img_folder, file_name)
        if not os.path.exists(src_img_path):
            continue
            
        new_file_name = f"tc_{file_name}"
        shutil.copy2(src_img_path, os.path.join(output_dir, 'images', 'test', new_file_name))
        
        txt_name = os.path.splitext(new_file_name)[0] + '.txt'
        dest_txt_path = os.path.join(output_dir, 'labels', 'test', txt_name)
        
        with open(dest_txt_path, 'w', encoding='utf-8') as f:
            for yolo_cls, bbox in anns:
                x_min, y_min, w, h = bbox
                xc = max(0.0, min(1.0, (x_min + w / 2.0) / meta['width']))
                yc = max(0.0, min(1.0, (y_min + h / 2.0) / meta['height']))
                nw = max(0.0, min(1.0, w / meta['width']))
                nh = max(0.0, min(1.0, h / meta['height']))
                f.write(f"{yolo_cls} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}\n")
                test_boxes_count += 1
                
        test_images_count += 1

# 4. Generate data.yaml
yaml_content = """# MarineGuard AI - YOLO11 Dataset Configuration
path: ./MarineGuard_YOLO_Ready
train: images/train
val: images/val
test: images/test

nc: 2
names:
  0: Net
  1: Rope
"""

yaml_path = os.path.join(output_dir, 'data.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content)

print(f"External test set (TrashCan) prepared: {test_images_count} images, {test_boxes_count} bounding boxes.")
print(f"Saved config at: {yaml_path}")

In [ ]:
import os

# Path to prepared dataset directory
dataset_dir = r"D:\01_University_&_Study\Samsung Innovation Campus - Immersive with MiSK Skills\CapstoneProject\MarineGuard_Datasets\MarineGuard_YOLO_Ready"
splits = ['train', 'val', 'test']

print("🔍 Starting comprehensive dataset integrity check...\n" + "="*50)

all_passed = True

for split in splits:
    img_dir = os.path.join(dataset_dir, 'images', split)
    lbl_dir = os.path.join(dataset_dir, 'labels', split)
    
    # Extract base filenames without extensions
    images = {os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))}
    labels = {os.path.splitext(f)[0] for f in os.listdir(lbl_dir) if f.endswith('.txt')}
    
    # 1. Verify 1-to-1 matching between images and label files
    missing_labels = images - labels
    orphan_labels = labels - images
    
    # 2. Validate YOLO label format and coordinate bounds [0, 1]
    corrupt_files = []
    empty_files = 0
    total_boxes = 0
    
    for lbl_name in labels:
        lbl_path = os.path.join(lbl_dir, lbl_name + '.txt')
        with open(lbl_path, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f.readlines() if line.strip()]
            
        if len(lines) == 0:
            empty_files += 1  # Expected for hard-negative background frames
            continue
            
        for line in lines:
            parts = line.split()
            # Each line must have 5 values: class_id xc yc w h
            if len(parts) != 5:
                corrupt_files.append(lbl_name)
                break
            try:
                cls_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]
                # Ensure class ID is 0 or 1 and coordinates are within [0.0, 1.0]
                if cls_id not in (0, 1) or any(c < 0.0 or c > 1.0 for c in coords):
                    corrupt_files.append(lbl_name)
                    break
                total_boxes += 1
            except ValueError:
                corrupt_files.append(lbl_name)
                break

    # Print split validation report
    print(f"📁 Split [{split.upper()}]:")
    print(f"   - Total Images: {len(images)} | Total TXT Labels: {len(labels)}")
    print(f"   - Valid Bounding Boxes: {total_boxes} | Empty Label Files (Hard-Negatives): {empty_files}")
    
    if missing_labels or orphan_labels or corrupt_files:
        all_passed = False
        if missing_labels: print(f"   ❌ Images missing TXT labels: {len(missing_labels)}")
        if orphan_labels:  print(f"   ❌ Orphan TXT labels without images: {len(orphan_labels)}")
        if corrupt_files:  print(f"   ❌ Malformed label files: {corrupt_files[:5]}")
    else:
        print("   ✅ Status: 100% Valid (Exact 1-to-1 match and valid YOLO format).")
    print("-" * 50)

if all_passed:
    print("\n🎉 Final Result: Dataset directory is verified and ready!")
else:
    print("\n⚠️ Please review the warnings above before proceeding.")

In [ ]:
import os
import yaml

# 1. Define dataset path
dataset_dir = r"D:\01_University_&_Study\Samsung Innovation Campus - Immersive with MiSK Skills\CapstoneProject\MarineGuard_Datasets\MarineGuard_YOLO"
yaml_path = os.path.join(dataset_dir, 'data.yaml')

# 2. Load class names from data.yaml
with open(yaml_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
class_names = config.get('names', {0: 'Net', 1: 'Rope'})

print("=" * 65)
print("🌊 MarineGuard AI — Final Dataset Summary Report (YOLO11 Ready)")
print("=" * 65)
print(f"Target Classes: {class_names}\n")
print(f"{'Split':<10} | {'Total Imgs':<12} | {'Net Boxes (0)':<15} | {'Rope Boxes (1)':<15} | {'Hard-Negs':<10}")
print("-" * 65)

totals = {'imgs': 0, 'net': 0, 'rope': 0, 'neg': 0}

# 3. Compute per-split statistics (train, val, test)
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(dataset_dir, 'images', split)
    lbl_dir = os.path.join(dataset_dir, 'labels', split)
    
    imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    net_count, rope_count, neg_count = 0, 0, 0
    
    for img_file in imgs:
        txt_file = os.path.splitext(img_file)[0] + '.txt'
        txt_path = os.path.join(lbl_dir, txt_file)
        
        if os.path.exists(txt_path):
            with open(txt_path, 'r', encoding='utf-8') as f:
                lines = [l.strip() for l in f.readlines() if l.strip()]
                
            if len(lines) == 0:
                neg_count += 1
            else:
                for line in lines:
                    cls_id = int(line.split()[0])
                    if cls_id == 0:
                        net_count += 1
                    elif cls_id == 1:
                        rope_count += 1

    totals['imgs'] += len(imgs)
    totals['net'] += net_count
    totals['rope'] += rope_count
    totals['neg'] += neg_count
    
    print(f"{split.upper():<10} | {len(imgs):<12} | {net_count:<15} | {rope_count:<15} | {neg_count:<10}")

print("-" * 65)
print(f"{'TOTAL':<10} | {totals['imgs']:<12} | {totals['net']:<15} | {totals['rope']:<15} | {totals['neg']:<10}")
print("=" * 65)
print("✅ Integrity Status: 100% Verified (Site-Aware Split & Normalized YOLO Format)")